# stockml on Kaggle: `stage.ipynb`

One notebook, four stages, driven by the `STAGE` variable in the next cell:

| `STAGE` | runs | needs a previous run attached as input? |
|---|---|---|
| `evolve` | `stockml evolve --config CONFIG_PATH` (fresh run) | no |
| `resume` | `stockml evolve --config CONFIG_PATH --resume <run dir>` | yes |
| `random` | `stockml evolve-control --kind random --run-dir <run dir>` | yes (a **completed** evolution run) |
| `null`   | `stockml evolve-control --kind null --run-dir <run dir>` | yes (a **completed** evolution run) |

**Kaggle notebook sessions cap out at 12 hours.** `configs/evo.yaml`'s default (`population_size: 40`, `generations: 25`, full S&P 500) will not finish in one session -- that's expected, not a bug to work around by shrinking the config (see CLAUDE.md: don't tune GA parameters against results). The intended chain, across separate committed notebook sessions, is:

1. **`evolve`** -- starts the run. When the session cap kills it mid-way, `progress.json` / `lineage.jsonl` already on disk are all `--resume` needs; nothing is lost.
2. **`resume`** (repeat as many times as it takes) -- each session picks up from the last completed generation until `progress.json` says `"status": "completed"`.
3. **`random`** and **`null`** -- once the evolution run is complete, run both controls against it (same budget each; can run in either order, in separate sessions if one alone doesn't fit in 12 hours either).
4. Only then: `stockml vault` -- run **locally**, not on Kaggle. The vault is a one-time look, guarded by `progress.json` saying `"status": "completed"`; there's no reason to spend a Kaggle session on a command that reads a few rows and writes a log entry.

Full details, including how a run's output moves between Kaggle sessions and back to your machine: README's "Running on Kaggle".

## Configure

The only cell you should need to edit between sessions. Set `STAGE`, point `PREV_RUN_INPUT_SLUG`/`RUN_NAME` at whichever run you're continuing (leave them alone for a fresh `evolve`), and fill in your own dataset slugs.

In [ ]:
# ============================== CONFIGURE ME ===============================
STAGE = "evolve"  # one of: "evolve" | "resume" | "random" | "null"

REPO_URL = "https://github.com/<you>/stockml.git"
REPO_REF = "main"                 # branch, tag, or commit to check out
CONFIG_PATH = "configs/evo.yaml"  # relative to the repo root

# From `python scripts/upload_cache_dataset.py` -- the price-cache dataset,
# attached to this notebook via Add Input.
CACHE_DATASET_SLUG = "yourusername/stockml-price-cache"

# Only read when STAGE is "resume" | "random" | "null": the Kaggle input
# holding a previous run's output -- typically this same notebook's own
# prior committed version, attached via Add Input -> Notebook Output --
# and that run's folder name (e.g. "evo_20260904_010203_evo"). Find the
# folder name with `stockml evo-report` or by listing runs/ after
# `stockml fetch-run` pulls it down locally.
PREV_RUN_INPUT_SLUG = "yourusername/stockml-stage-evolve"
RUN_NAME = "evo_20260904_010203_evo"

# Optional, independent of STAGE above -- see "Optional: refresh the
# price-cache dataset" near the end of this notebook. Needs the
# "Authenticate the kaggle CLI" cell below to have real Secrets behind it.
REFRESH_PRICE_CACHE = False
# =============================================================================

## Clone the repo

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/stockml")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"[stage] {REPO_DIR} already exists -- reusing (delete it first to re-clone)")
%cd {REPO_DIR}

## Authenticate the `kaggle` CLI (from Kaggle Secrets)

Only needed for the optional price-cache refresh/upload near the end of this notebook -- the main stages below (`evolve`/`resume`/`random`/`null`) never call the `kaggle` CLI themselves, they just read the mounted input dataset. Add two Secrets to this notebook first (Add-ons -> Secrets): `KAGGLE_USERNAME` and `KAGGLE_KEY`, the same two fields as your local `~/.kaggle/kaggle.json` (see https://www.kaggle.com/settings -> API -> Create New Token, which downloads that file). This cell is harmless to run even if you don't set `REFRESH_PRICE_CACHE = True` -- it just won't be used.

In [ ]:
import os
# kaggle ships pre-installed on Kaggle's own notebook image; -q upgrade
# here only guards against a stale pre-installed version.
!pip install -q -U kaggle

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
    print("[stage] kaggle CLI authenticated from Kaggle Secrets (KAGGLE_USERNAME/KAGGLE_KEY)")
except Exception as e:
    print(
        f"[stage][WARN] could not read KAGGLE_USERNAME/KAGGLE_KEY from Secrets ({e}) -- "
        f"the kaggle CLI won't be authenticated. Fine unless REFRESH_PRICE_CACHE=True below."
    )

## Install pinned dependencies

In [ ]:
!pip install -q -r kaggle/requirements.txt
!pip install -q -e .

## Point the price cache and run output at the right places

Symlinks `data/cache` onto the mounted (read-only) cache dataset, and sets `STOCKML_CACHE_DIR`/`STOCKML_RUNS_DIR` so `configs/evo.yaml` doesn't need editing -- see `run.apply_env_overrides` in the repo.

In [ ]:
import os
from pathlib import Path

cache_target = Path("data/cache")
cache_target.parent.mkdir(parents=True, exist_ok=True)
cache_source = Path(f"/kaggle/input/{CACHE_DATASET_SLUG.split('/')[-1]}")
if not cache_source.is_dir():
    raise SystemExit(
        f"{cache_source} not found -- attach the '{CACHE_DATASET_SLUG}' dataset "
        f"to this notebook (Add Input) before running this cell."
    )
if cache_target.is_symlink():
    cache_target.unlink()
elif cache_target.exists():
    raise SystemExit(f"{cache_target} exists and is a real directory -- refusing to remove it")
cache_target.symlink_to(cache_source)
print(f"[stage] {cache_target} -> {cache_source}")

os.environ["STOCKML_CACHE_DIR"] = str(cache_target.resolve())
os.environ["STOCKML_RUNS_DIR"] = "/kaggle/working/runs"
os.environ["PYTHONUNBUFFERED"] = "1"
Path(os.environ["STOCKML_RUNS_DIR"]).mkdir(parents=True, exist_ok=True)
print(f"[stage] STOCKML_CACHE_DIR={os.environ['STOCKML_CACHE_DIR']}")
print(f"[stage] STOCKML_RUNS_DIR={os.environ['STOCKML_RUNS_DIR']}")

## Bring in a previous run (resume / random / null only)

Copies `RUN_NAME`'s folder from the attached previous-run input into `/kaggle/working/runs`, where `STOCKML_RUNS_DIR` now points -- `stockml evolve --resume` / `evolve-control --run-dir` need a real, writable copy, not the read-only input mount itself.

In [ ]:
import shutil
from pathlib import Path

RUN_DIR = Path(os.environ["STOCKML_RUNS_DIR"]) / RUN_NAME

if STAGE in ("resume", "random", "null"):
    prev_input = Path(f"/kaggle/input/{PREV_RUN_INPUT_SLUG.split('/')[-1]}") / "runs" / RUN_NAME
    if not prev_input.is_dir():
        raise SystemExit(
            f"{prev_input} not found -- attach the previous run's output as an input "
            f"(Add Input -> Notebook Output, or a dataset built via `stockml fetch-run`), "
            f"and check RUN_NAME matches its runs/<...>/ folder name."
        )
    if not RUN_DIR.exists():
        shutil.copytree(prev_input, RUN_DIR)
        print(f"[stage] copied {prev_input} -> {RUN_DIR}")
    else:
        print(f"[stage] {RUN_DIR} already present in this session -- not re-copying")
else:
    print("[stage] fresh evolve run -- nothing to copy")

## Run the stage

In [ ]:
import subprocess

if STAGE == "evolve":
    cmd = ["stockml", "evolve", "--config", CONFIG_PATH]
elif STAGE == "resume":
    cmd = ["stockml", "evolve", "--config", CONFIG_PATH, "--resume", str(RUN_DIR)]
elif STAGE == "random":
    cmd = ["stockml", "evolve-control", "--config", CONFIG_PATH, "--kind", "random", "--run-dir", str(RUN_DIR)]
elif STAGE == "null":
    cmd = ["stockml", "evolve-control", "--config", CONFIG_PATH, "--kind", "null", "--run-dir", str(RUN_DIR)]
else:
    raise SystemExit(f"unknown STAGE {STAGE!r} -- must be one of evolve/resume/random/null")

print(f"[stage] running: {' '.join(cmd)}")
result = subprocess.run(cmd, env={**os.environ, "PYTHONUNBUFFERED": "1"})
if result.returncode != 0:
    raise SystemExit(f"[stage] {STAGE} exited with code {result.returncode}")
print(f"[stage] {STAGE} finished")

## After this cell finishes

**Commit this notebook version** (Save Version -> Save & Run All) so `/kaggle/working` -- including `runs/`, whether the stage above ran to completion or was cut off by the 12-hour cap -- becomes this version's attachable output. Then either:

- continue the chain on Kaggle: add this notebook's own output as an input to itself (or a copy), set `STAGE`/`RUN_NAME` for the next step above, and run again; or
- pull the result down locally: `stockml fetch-run <your-username>/<this-notebook-slug>`.

In [ ]:
from pathlib import Path
import json

runs_root = Path(os.environ["STOCKML_RUNS_DIR"])
evo_dirs = [p for p in runs_root.glob("evo_*") if p.is_dir()]
if evo_dirs:
    latest = max(evo_dirs, key=lambda p: p.stat().st_mtime)
    print(f"[stage] latest run dir: {latest}")
    progress_path = latest / "progress.json"
    if progress_path.exists():
        print(json.dumps(json.loads(progress_path.read_text()), indent=2))
else:
    print(f"[stage] no evo_* run dirs under {runs_root} yet")

## Optional: refresh the price-cache dataset

A separate, self-contained maintenance action -- normally its own session, not combined with a STAGE run (a dataset version you just pushed does not remount into *this* session; the benefit shows up the *next* time this notebook -- or a fresh one -- attaches CACHE_DATASET_SLUG as input). Only runs if `REFRESH_PRICE_CACHE = True` in the Configure cell (default False, so it is a no-op otherwise); needs the `kaggle` CLI authenticated (the cell above) with **write** access to `CACHE_DATASET_SLUG`, and assumes that dataset already exists (this always pushes a *version*, i.e. `kaggle datasets version` -- for the very first upload, see README's "One-time setup: upload the price cache", run locally).

Downloads into `data/cache_fresh/` -- not `data/cache`, which the "Point the price cache" cell above symlinks onto the read-only mounted input -- then calls `scripts/upload_cache_dataset.py --version`, the same script that one-time local setup runs.


In [ ]:
import subprocess

if REFRESH_PRICE_CACHE:
    FRESH_CACHE_DIR = "data/cache_fresh"
    kaggle_username, dataset_slug = CACHE_DATASET_SLUG.split("/")

    print(f"[stage] downloading fresh prices into {FRESH_CACHE_DIR} ...")
    subprocess.run(
        ["stockml", "download", "--config", CONFIG_PATH],
        env={**os.environ, "STOCKML_CACHE_DIR": FRESH_CACHE_DIR, "PYTHONUNBUFFERED": "1"},
        check=True,
    )

    print(f"[stage] pushing a new version of {CACHE_DATASET_SLUG} ...")
    subprocess.run(
        [
            "python", "scripts/upload_cache_dataset.py",
            "--cache-dir", FRESH_CACHE_DIR,
            "--kaggle-username", kaggle_username,
            "--slug", dataset_slug,
            "--version",
            "--version-notes", "Refreshed from a Kaggle notebook run.",
        ],
        check=True,
    )
    print("[stage] price-cache dataset refreshed")
else:
    print("[stage] REFRESH_PRICE_CACHE is False -- skipping")